# IslamicEval 2026 — Task 2 · Experiments runner (Colab, saves to HF)

Runs the full **hallucination-identification** pipeline end-to-end and pushes every result to your
Hugging Face repo, so the numbers can be pulled straight into the paper. Fast on Colab (a few minutes;
CPU is enough — the core is TF-IDF + fuzzy matching. An optional embedding backend can use the GPU).

It produces and saves:
1. the submitted-system dev result (per-type + macro) scored by the official scorer;
2. the ablation (attribution-as-text → parent-linked → grounded isnad);
3. a retrieval-backend comparison (char-TFIDF vs word-TFIDF vs BM25, optional embeddings);
4. per-type misclassified development examples (with the Arabic span + nearest source);
5. the dev submission TSV/zip.

**Setup:** add your token to Colab **Secrets** (🔑) as `HF_TOKEN`. GPU runtime only needed if you set
`USE_EMBED=True`.

## 0 · Deps + HF auth + clone

In [ ]:
!pip -q install rapidfuzz scikit-learn rank_bm25 huggingface_hub pandas numpy
import os, sys, json, subprocess, re
from pathlib import Path
from huggingface_hub import login, HfApi
HF_USER = "FatimahEmadEldin"
HF_DATASET = f"{HF_USER}/IslamicEval2026-Subtask2-Submission"
try:
    from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets (key icon)."
login(HF_TOKEN); API = HfApi()
REPO = Path("/content/IslamicEval2026")
if not REPO.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/Watheq9/IslamicEval2026.git",str(REPO)],check=True)
QURAN_PATH=REPO/"Corpora/quranic_verses.json"; HADITH_PATH=REPO/"Corpora/six_hadith_books.json"
DEV=REPO/"dev_set/dev.jsonl"; TRAIN=REPO/"train_set/train.jsonl"
GOLD=REPO/"dev_set/dev_task_2.tsv"; SCORER=REPO/"Scoring_scripts/task2_scoring.py"
USE_EMBED = False   # set True on a GPU runtime to add a multilingual-embedding backend
print("ready:", all(p.exists() for p in [QURAN_PATH,HADITH_PATH,DEV,TRAIN,GOLD,SCORER]))

## 1 · Normalization (Arabic ranges from codepoints) + loaders

In [ ]:
_T=[(0x610,0x61A),(0x64B,0x65F),(0x670,0x670),(0x6D6,0x6DC),(0x6DF,0x6E8),(0x6EA,0x6ED)]
_TASHKEEL=re.compile('['+''.join(chr(a)+'-'+chr(b) for a,b in _T)+']'); _TAT=chr(0x640)
_NON_AR=re.compile('[^'+chr(0x621)+'-'+chr(0x64A)+'\\s]'); _SP=re.compile(r'\s+')
_ALEF=re.compile('['+''.join(chr(c) for c in (0x622,0x623,0x625,0x627,0x671,0x621))+']')
def normalize(t):
    if not t: return ""
    t=_SP.sub(' ',_TASHKEEL.sub('',str(t)).replace(_TAT,'')).strip()
    t=_ALEF.sub(chr(0x627),t).replace(chr(0x649),chr(0x64A)).replace(chr(0x624),chr(0x648)).replace(chr(0x626),chr(0x64A)).replace(chr(0x629),chr(0x647))
    return _SP.sub(' ',_NON_AR.sub(' ',t)).strip()
def rj(p):
    txt=Path(p).read_text(encoding='utf-8').strip()
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]
def fk(d,ks):
    for k in ks:
        if k in d and d[k] not in (None,""): return d[k]
def load_quran(p):
    o=[]
    for d in rj(p):
        t=fk(d,["ayah_text","text"])
        if t: o.append({"text":str(t),"norm":normalize(t),"surah_id":fk(d,["surah_id"]),"surah_name":fk(d,["surah_name"]),"ayah_id":fk(d,["ayah_id"])})
    return o
def load_hadith(p):
    o=[]
    for d in rj(p):
        m=fk(d,["Matn","matn","text"])
        if not m: continue
        full=fk(d,["hadithTxt"]) or ""; nm=normalize(m); nf=normalize(full)
        o.append({"text":str(m),"norm":nm,"book":fk(d,["title"]),"full_norm":nf,"chain_norm":(nf.replace(nm," ").strip() if nm and nm in nf else nf)})
    return o
def load_segments(p):
    data=rj(p); segs=[]
    for r in data:
        rid=fk(r,["id"]); ans=fk(r,["generated_answer"]) or ""
        for ann in r.get("annotations") or []:
            aid=fk(ann,["annotation_id","id"])
            for s in ann.get("segments") or []:
                a=s.get("span_start"); b=s.get("span_end"); txt=ans[a:b] if (a is not None and b is not None and b>a) else (s.get("span_text") or "")
                segs.append({"resp_id":rid,"ann_id":aid,"seg_type":s.get("type"),"span_text":txt,"gold":s.get("label")})
    return segs,data
import pandas as pd, numpy as np
QURAN=load_quran(QURAN_PATH); HADITH=load_hadith(HADITH_PATH)
dev,_=load_segments(DEV); train,_=load_segments(TRAIN)
keep=set(list(dict.fromkeys(s["resp_id"] for s in train))[:1200]); tune=[s for s in train if s["resp_id"] in keep]
print("quran",len(QURAN),"hadith",len(HADITH),"dev",len(dev),"tune",len(tune))

## 2 · Retrieval backends (char-TFIDF, word-TFIDF, BM25, optional embeddings)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel
from rapidfuzz import fuzz
import scipy.sparse as sp
def _rerank(qn, idxs, recs, topn):
    sc=[]
    for j in idxs:
        r=recs[j]; v=max(fuzz.token_set_ratio(qn,r["norm"]),fuzz.partial_ratio(qn,r["norm"]))/100.0; sc.append((v,r))
    sc.sort(key=lambda x:-x[0]); return (sc[0][0],sc[0][1],sc[:topn]) if sc else (0.0,None,[])
class TFIDF:
    def __init__(s,recs,analyzer,ngram,name):
        s.recs=recs; s.name=name; s.vec=TfidfVectorizer(analyzer=analyzer,ngram_range=ngram,min_df=1); s.mat=s.vec.fit_transform([r["norm"] for r in recs])
    def score_spans(s,spans,k=15,topn=1,chunk=256):
        qn=[normalize(x) for x in spans]; res=[(0.0,None,[]) for _ in spans]; idx=[i for i,q in enumerate(qn) if q]
        if not idx: return res
        Q=s.vec.transform([qn[i] for i in idx])
        for st in range(0,len(idx),chunk):
            sub=idx[st:st+chunk]; sims=linear_kernel(Q[st:st+chunk],s.mat)
            for row,i in enumerate(sub):
                kk=min(k,sims.shape[1]); top=np.argpartition(sims[row],-kk)[-kk:]; res[i]=_rerank(qn[i],top,s.recs,topn)
        return res
class BM25B:
    # Vectorised Okapi BM25: precompute the doc-term weight matrix W once, then score a whole
    # batch of queries with one sparse matmul (Q_binary @ W.T) instead of one query at a time.
    name="BM25"
    def __init__(s,recs,k1=1.5,b=0.75):
        s.recs=recs; s.cv=CountVectorizer(token_pattern=r"(?u)\b\w+\b")
        X=s.cv.fit_transform([r["norm"] for r in recs]).tocsr(); N,V=X.shape
        df=np.asarray((X>0).sum(0)).ravel(); idf=np.log(1+(N-df+0.5)/(df+0.5))
        dl=np.asarray(X.sum(1)).ravel(); avgdl=dl.mean() if dl.mean() else 1.0
        C=X.tocoo(); denom=C.data + k1*(1-b+b*dl[C.row]/avgdl)
        w=idf[C.col]*C.data*(k1+1)/denom
        s.W=sp.csr_matrix((w,(C.row,C.col)),shape=(N,V))
    def score_spans(s,spans,k=15,topn=1,chunk=256):
        qn=[normalize(x) for x in spans]; res=[(0.0,None,[]) for _ in spans]; idx=[i for i,q in enumerate(qn) if q]
        if not idx: return res
        Q=(s.cv.transform([qn[i] for i in idx])>0).astype(float)
        for st in range(0,len(idx),chunk):
            sub=idx[st:st+chunk]; sims=np.asarray((Q[st:st+chunk] @ s.W.T).todense())
            for row,i in enumerate(sub):
                kk=min(k,sims.shape[1]); top=np.argpartition(sims[row],-kk)[-kk:]; res[i]=_rerank(qn[i],top,s.recs,topn)
        return res
def char(recs): return TFIDF(recs,"char_wb",(3,5),"char-TFIDF (ours)")
def word(recs): return TFIDF(recs,"word",(1,2),"word-TFIDF")
print("backends defined")

## 3 · Verifiers, precompute, thresholding, metric

In [ ]:
SURAH={normalize(v["surah_name"]):v["surah_id"] for v in QURAN if v.get("surah_name") and v.get("surah_id") is not None}
AR2EN=str.maketrans(''.join(chr(0x660+i) for i in range(10)),'0123456789')
def find_number(t):
    m=re.search(r'\d+',str(t).translate(AR2EN)); return int(m.group()) if m else None
def _w(*c): return normalize(''.join(chr(x) for x in c))
BOOKS=[_w(0x627,0x644,0x628,0x62E,0x627,0x631,0x64A),_w(0x645,0x633,0x644,0x645),_w(0x627,0x644,0x62A,0x631,0x645,0x630,0x64A),
 _w(0x627,0x644,0x646,0x633,0x627,0x626,0x64A),_w(0x627,0x628,0x646,0x20,0x645,0x627,0x62C,0x647),_w(0x627,0x62D,0x645,0x62F),_w(0x645,0x627,0x644,0x643)]
def verify_cs_parent(span,pk,pr):
    c=normalize(span)
    if pr is None or not c: return "correct"
    if pk=="Ayah":
        sid=next((v for n,v in SURAH.items() if n and len(n)>2 and n in c),None)
        if sid is None: return "correct"
        if str(sid)!=str(pr.get("surah_id")): return "incorrect"
        n=find_number(span)
        if n is not None and pr.get("ayah_id") is not None: return "correct" if str(n)==str(pr.get("ayah_id")) else "incorrect"
        return "correct"
    cb=next((b for b in BOOKS if b in c),None); tb=normalize(str(pr.get("book") or ""))
    if cb is None or not tb: return "correct"
    return "correct" if (cb in tb or tb in cb) else "incorrect"
SEG_TYPES=["Ayah","matn","isnad","claimed_source"]
def macro(df):
    per={}
    for st in SEG_TYPES:
        sub=df[(df["seg_type"]==st)&(df["gold"].isin(["correct","incorrect"]))]
        per[st]=float((sub["pred"]==sub["gold"]).mean()) if len(sub) else float("nan")
    v=[x for x in per.values() if x==x]; per["MACRO"]=sum(v)/len(v) if v else float("nan"); return per
def precompute(segs,QB,HB,cs_mode="parent"):
    rows=[dict(s) for s in segs]; by={t:[i for i,s in enumerate(segs) if (s["seg_type"] or "").strip()==t] for t in SEG_TYPES}; parent={}
    for pos,(sc,rec,_) in zip(by["Ayah"], QB.score_spans([segs[i]["span_text"] for i in by["Ayah"]])):
        rows[pos].update(_score=sc,_rec=rec); parent[(segs[pos]["resp_id"],segs[pos]["ann_id"])]=("Ayah",rec,[rec])
    for pos,(sc,rec,top3) in zip(by["matn"], HB.score_spans([segs[i]["span_text"] for i in by["matn"]],topn=3)):
        rows[pos].update(_score=sc,_rec=rec); parent[(segs[pos]["resp_id"],segs[pos]["ann_id"])]=("matn",rec,[r for _,r in top3])
    cs_idx=by["claimed_source"]; cs_txts=[segs[i]["span_text"] for i in cs_idx]
    qa=QB.score_spans(cs_txts) if cs_txts else []; ha=HB.score_spans(cs_txts) if cs_txts else []
    for pos,(sa,_,_),(sh,_,_) in zip(cs_idx,qa,ha):
        pk,pr,_=parent.get((segs[pos]["resp_id"],segs[pos]["ann_id"]),(None,None,[]))
        rows[pos].update(_cs=verify_cs_parent(segs[pos]["span_text"],pk,pr),_cs_astext=max(sa,sh),_rec=pr)
    for pos in by["isnad"]:
        pk,pr,tops=parent.get((segs[pos]["resp_id"],segs[pos]["ann_id"]),(None,None,[])); q=normalize(segs[pos]["span_text"]); fs=0.0; best=None
        if q and pk=="matn":
            for r in tops:
                if r:
                    vv=max(fuzz.token_set_ratio(q,r.get("full_norm","")),fuzz.partial_ratio(q,r.get("full_norm","")))/100.0
                    if vv>fs: fs,best=vv,r
        rows[pos].update(_isnad=fs,_rec=best)
    for r in rows: r.setdefault("_score",0.0); r.setdefault("_cs","incorrect"); r.setdefault("_isnad",0.0); r.setdefault("_rec",None)
    return rows
def apply_(rows,ta,tm,ti,isnad_mode="grounded",cs_astext=False):
    o=[]
    for r in rows:
        st=r["seg_type"]
        if st=="Ayah": p="correct" if r["_score"]>=ta else "incorrect"
        elif st=="matn": p="correct" if r["_score"]>=tm else "incorrect"
        elif st=="claimed_source": p=("correct" if r.get("_cs_astext",0)>=tm else "incorrect") if cs_astext else r["_cs"]
        elif st=="isnad": p=("correct" if r["_isnad"]>=ti else "incorrect") if isnad_mode=="grounded" else "correct"
        else: p="incorrect"
        o.append({**r,"pred":p})
    return pd.DataFrame(o)
def tune_taus(rows,isnad_mode,ti=0.85):
    best=-1;bc=(0.9,0.82)
    for ta in [round(x,2) for x in np.arange(0.80,0.99,0.02)]:
        for tm in [round(x,2) for x in np.arange(0.70,0.95,0.02)]:
            m=macro(apply_(rows,ta,tm,ti,isnad_mode))["MACRO"]
            if m>best: best,bc=m,(ta,tm)
    return bc
print("verifiers ready")

## 4 · Submitted system: dev result + official score

In [ ]:
QC,HC=char(QURAN),char(HADITH)
tr=precompute(tune,QC,HC,"parent"); dr=precompute(dev,QC,HC,"parent")
TA,TM=tune_taus(tr,"grounded"); TI=0.85
pred=apply_(dr,TA,TM,TI,"grounded")
m=macro(pred); print("dev per-type:",{k:round(v,3) for k,v in m.items()},"taus",(TA,TM,TI))
sub=pred[["resp_id","ann_id","seg_type","pred"]].copy(); sub.columns=["Response_ID","Annotation_ID","Segment_Type","Label"]
sub=sub[sub["Label"].isin(["correct","incorrect"])].drop_duplicates(subset=["Response_ID","Annotation_ID","Segment_Type"])
OUT="/content/submission_task2_dev.tsv"; sub.to_csv(OUT,sep="\t",index=False)
o=Path("/content/score"); o.mkdir(exist_ok=True)
r=subprocess.run([sys.executable,str(SCORER),"--pred",OUT,"--ref",str(GOLD),"--output",str(o),"-v"],capture_output=True,text=True)
official=json.loads((o/"scores.json").read_text()); print("OFFICIAL:",official)
RESULTS={"dev_official":official,"taus":{"tau_ayah":TA,"tau_matn":TM,"tau_isnad":TI}}

## 5 · Ablation (attribution-as-text → parent-linked → grounded isnad)

In [ ]:
abl=[]
# A: attribution as text (predict prior 'correct'), isnad prior
ta,tm=tune_taus(tr,"prior"); a=macro(apply_(dr,ta,tm,TI,"prior",cs_astext=True)); a["config"]="attribution as text + isnad prior"; abl.append(a)
# B: parent-linked attribution, isnad prior
b=macro(apply_(dr,ta,tm,TI,"prior")); b["config"]="+ parent-linked attribution"; abl.append(b)
# C: parent-linked + grounded isnad (submitted)
c=macro(apply_(dr,TA,TM,TI,"grounded")); c["config"]="+ grounded isnad (submitted)"; abl.append(c)
abl_df=pd.DataFrame(abl)[["config"]+SEG_TYPES+["MACRO"]]; print(abl_df.round(3).to_string(index=False))
RESULTS["ablation"]=abl

## 6 · Retrieval-backend comparison (char-TFIDF vs word-TFIDF vs BM25 [+ embeddings])

In [ ]:
def run_backend(QB,HB,label):
    trb=precompute(tune,QB,HB,"parent"); drb=precompute(dev,QB,HB,"parent")
    ta,tm=tune_taus(trb,"grounded"); mm=macro(apply_(drb,ta,tm,TI,"grounded")); mm=dict(mm); mm["backend"]=label; mm["taus"]=[ta,tm]
    print(label,{k:round(v,3) for k,v in mm.items() if k in SEG_TYPES+['MACRO']}); return mm
comp=[]
comp.append(run_backend(QC,HC,"char-TFIDF (ours)"))
comp.append(run_backend(word(QURAN),word(HADITH),"word-TFIDF"))
try: comp.append(run_backend(BM25B(QURAN),BM25B(HADITH),"BM25"))
except Exception as e: print("BM25 skipped:",e)
if USE_EMBED:
    try:
        !pip -q install sentence-transformers
        from sentence_transformers import SentenceTransformer
        _m=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cuda")
        class EMB:
            def __init__(s,recs): s.recs=recs; s.emb=_m.encode([r["norm"] for r in recs],convert_to_numpy=True,normalize_embeddings=True,batch_size=256,show_progress_bar=True)
            def score_spans(s,spans,k=15,topn=1,chunk=256):
                qn=[normalize(x) for x in spans]; res=[(0.0,None,[]) for _ in spans]; idx=[i for i,q in enumerate(qn) if q]
                if not idx: return res
                qe=_m.encode([qn[i] for i in idx],convert_to_numpy=True,normalize_embeddings=True,batch_size=256)
                sims=qe@s.emb.T
                for row,i in enumerate(idx):
                    kk=min(k,sims.shape[1]); top=np.argpartition(sims[row],-kk)[-kk:]; res[i]=_rerank(qn[i],top,s.recs,topn)
                return res
        comp.append(run_backend(EMB(QURAN),EMB(HADITH),"MiniLM embeddings (GPU)"))
    except Exception as e: print("embeddings skipped:",e)
comp_df=pd.DataFrame(comp)[["backend"]+SEG_TYPES+["MACRO"]]; print(comp_df.round(3).to_string(index=False))
RESULTS["backend_comparison"]=comp

## 7 · Misclassified development examples (per type) + LaTeX fragment

In [ ]:
def esc(t):
    t=str(t).replace("\n"," ").replace("\\","")
    for a,b in [("&","\\&"),("%","\\%"),("_","\\_"),("#","\\#"),("$","\\$"),("{","\\{"),("}","\\}"),("~"," "),("^"," ")]: t=t.replace(a,b)
    return t.strip()
def trunc(t,n=55):
    t=str(t).strip(); return t[:n]+("\\ldots" if len(t)>n else "")
rows_ex=[]; frag=["\\begin{table}[h]\n\\centering\\small\n\\setlength{\\tabcolsep}{4pt}\n\\begin{tabular}{@{}llp{3.1cm}p{3.1cm}@{}}\n\\toprule",
"\\textbf{Type} & \\textbf{gold/pred} & \\textbf{quoted span} & \\textbf{nearest source} \\\\\n\\midrule"]
for st in SEG_TYPES:
    subm=pred[(pred["seg_type"]==st)&(pred["gold"].isin(["correct","incorrect"]))&(pred["pred"]!=pred["gold"])]
    subm=subm[subm["span_text"].str.len()>8]
    for _,r in subm.head(1).iterrows():
        rec=r.get("_rec") or {}; srctxt=rec.get("text","") if isinstance(rec,dict) else ""
        lbl="claimed src" if st=="claimed_source" else st
        rows_ex.append({"type":st,"gold":r["gold"],"pred":r["pred"],"span":r["span_text"],"nearest_source":srctxt})
        frag.append(f"{lbl} & {r['gold']}/{r['pred']} & \\ar{{{esc(trunc(r['span_text']))}}} & \\ar{{{esc(trunc(srctxt))}}} \\\\\n\\addlinespace[2pt]")
frag.append("\\bottomrule\n\\end{tabular}\n\\caption{Representative development misclassifications, one per segment type.}\n\\label{tab:errors}\n\\end{table}")
open("/content/examples_table.tex","w",encoding="utf-8").write("\n".join(frag))
pd.DataFrame(rows_ex).to_csv("/content/misclassified_examples.tsv",sep="\t",index=False)
RESULTS["misclassified_examples"]=rows_ex
print("examples written:",len(rows_ex))

## 8 · Save everything to your HF repo

In [ ]:
import zipfile
json.dump(RESULTS, open("/content/results.json","w"), ensure_ascii=False, indent=2)
comp_df.to_csv("/content/backend_comparison.tsv",sep="\t",index=False)
abl_df.to_csv("/content/ablation.tsv",sep="\t",index=False)
with zipfile.ZipFile("/content/submission_task2_dev.zip","w",zipfile.ZIP_DEFLATED) as zf: zf.write(OUT,"submission_task2_dev.tsv")
uploads=[("/content/results.json","experiments/results.json"),
         ("/content/ablation.tsv","experiments/ablation.tsv"),
         ("/content/backend_comparison.tsv","experiments/backend_comparison.tsv"),
         ("/content/misclassified_examples.tsv","experiments/misclassified_examples.tsv"),
         ("/content/examples_table.tex","experiments/examples_table.tex"),
         ("/content/submission_task2_dev.tsv","experiments/submission_task2_dev.tsv"),
         ("/content/submission_task2_dev.zip","experiments/submission_task2_dev.zip")]
for lo,re_ in uploads:
    API.upload_file(path_or_fileobj=lo,path_in_repo=re_,repo_id=HF_DATASET,repo_type="dataset"); print("uploaded",re_)
print("\\nAll results saved to https://huggingface.co/datasets/"+HF_DATASET+"/tree/main/experiments")